In [1]:
!pip install fake_useragent

In [36]:
import requests
import json
from fake_useragent import UserAgent

# 職缺 ID（可換）
job_id = "8jvge"

# 偽造瀏覽器標頭
headers = {
    "User-Agent": UserAgent().random,
    "Referer": f"https://www.104.com.tw/job/{job_id}"
}

# 取得 JSON 資料
url = f"https://www.104.com.tw/job/ajax/content/{job_id}"
res = requests.get(url, headers=headers)

if res.status_code == 200:
    job_json = res.json()
    
    # 儲存成漂亮的 JSON 檔案（有階層結構）
    with open(f"104_job_{job_id}.json", "w", encoding="utf-8") as f:
        json.dump(job_json, f, ensure_ascii=False, indent=2)

    print(f"✅ 已儲存 job_id={job_id} 的完整 JSON 到檔案！")
else:
    print(f"❌ 抓取失敗：HTTP {res.status_code}")

✅ 已儲存 job_id=8jvge 的完整 JSON 到檔案！


In [ ]:
import requests
import time
import random
import pandas as pd
from fake_useragent import UserAgent
from datetime import datetime, timedelta

today = datetime.today()
one_week_ago = today - timedelta(days=7)

# 中文職缺對照表
job_categories = {
    "2007001015": "前端工程師",
    "2007001018": "數據分析師",
    "2007001020": "AI工程師",
    "2007002005": "網路管理工程師",
    "2007002010": "雲端工程師"
}

# 嘗試讀取已存在檔案，取得已存在的職缺連結
existing_links = set()
try:
    existing_df = pd.read_csv("104_五類工程師職缺_0411.csv", sep="|", encoding="utf-8-sig")
    existing_links = set(existing_df["link"].dropna())
    print(f"🔎 已載入舊資料，共有 {len(existing_links)} 筆職缺連結")
except FileNotFoundError:
    print("📂 未找到舊檔案，將建立新的檔案")
    existing_df = pd.DataFrame()

# 抓取職缺詳細資料
def get_job_detail(job_id, headers, job_type_name):
    url = f"https://www.104.com.tw/job/ajax/content/{job_id}"

    for _ in range(3):
        res = requests.get(url, headers=headers)
        if res.status_code == 429:
            time.sleep(random.uniform(5, 10))
            continue
        break

    try:
        data = res.json().get("data", {})
    except Exception:
        return None

    salary = data.get("jobDetail", {}).get("salary", "")
    if "月薪" not in salary and "年薪" not in salary:
        return None

    if data.get("jobDetail", {}).get("jobType", None) != 1:
        return None

    appear_date = data.get("header", {}).get("appearDate", "")
    if appear_date:
        appear_dt = datetime.strptime(appear_date, "%Y/%m/%d")
        if appear_dt < one_week_ago:
            return None
        appear_date = appear_dt.strftime("%Y-%m-%d")

    # 檢查 header 中是否包含「實習」
    job_name = data.get("header", {}).get("jobName", "")
    if "實習" in job_name:
        return None
    
    return {
        "jobType": job_type_name,
        "header": data.get("header", {}).get("jobName", ""),
        "jobDetail": data.get("jobDetail", {}).get("jobDescription", ""),
        "salary": salary,
        "workExp": data.get("condition", {}).get("workExp", ""),
        "edu": data.get("condition", {}).get("edu", ""),
        "tool": ", ".join([s.get("description", "") for s in data.get("condition", {}).get("specialty", [])]),
        "welfare": data.get("welfare", {}).get("welfare", ""),
        "area": data.get("jobDetail", {}).get("addressRegion", ""),
        "date": appear_date,
        "link": f"https://www.104.com.tw/job/{job_id}?jobsource=m_index_s",
    }

# 抓取職缺 ID 並篩選符合條件的
def get_jobs_for_category(jobcat_code, jobcat_name):
    user_agent = UserAgent()
    job_list = []
    page = 1
    duplicate_count = 0  # 新增重複計數器

    while page <= 100 and len(job_list) < 20:
        headers = {
            "User-Agent": user_agent.random,
            "Referer": "https://www.104.com.tw/jobs/search/?",
        }

        print(f"🔍 搜尋 {jobcat_name} 第 {page} 頁")
        url = "https://www.104.com.tw/jobs/search/list"
        params = {
            "ro": "0",  # 全職
            "jobcat": jobcat_code,
            "page": page,
            "mode": "s",
            "jobsource": "2018indexpoc"
        }

        res = requests.get(url, headers=headers, params=params)
        job_data = res.json().get("data", {}).get("list", [])

        if not job_data:
            print(f"❌ {jobcat_name} 第 {page} 頁無資料")
            break

        for job in job_data:
            if len(job_list) >= 20:
                break

            job_url = job.get("link", {}).get("job", "")
            job_id = job_url.split("/job/")[-1].split("?")[0]
            job_link = f"https://www.104.com.tw/job/{job_id}?jobsource=m_index_s"

            if job_link in existing_links:
                duplicate_count += 1
                print(f"⏭️ 跳過重複職缺：{job_link}（連續第 {duplicate_count} 筆）")
                if duplicate_count >= 5:
                    print(f"🛑 {jobcat_name} 遇到連續 5 筆重複，停止該類爬取")
                    return pd.DataFrame(job_list)
                continue
            else:
                duplicate_count = 0  # 有新資料就重置重複計數器

            detail_info = get_job_detail(job_id, headers, jobcat_name)
            if detail_info:
                job_list.append(detail_info)
                existing_links.add(job_link)

        page += 1
        time.sleep(random.uniform(3, 6))

    print(f"✅ {jobcat_name} 完成，共抓到 {len(job_list)} 筆新職缺")
    return pd.DataFrame(job_list)

# 主程式
all_jobs = pd.DataFrame()

for jobcat_code, jobcat_name in job_categories.items():
    df = get_jobs_for_category(jobcat_code, jobcat_name)
    all_jobs = pd.concat([all_jobs, df], ignore_index=True)

# 合併舊資料與新資料
if not all_jobs.empty:
    final_df = pd.concat([existing_df, all_jobs], ignore_index=True)
else:
    final_df = existing_df

# 儲存成｜分隔的 CSV
final_df.to_csv("jobs.csv", index=False, sep="|", encoding="utf-8-sig")


🔎 已載入舊資料，共有 979 筆職缺連結
🔍 搜尋 前端工程師 第 1 頁
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8omdz?jobsource=m_index_s（連續第 1 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8omcw?jobsource=m_index_s（連續第 2 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8nwn1?jobsource=m_index_s（連續第 1 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8kl7m?jobsource=m_index_s（連續第 1 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8lozn?jobsource=m_index_s（連續第 2 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/67xqd?jobsource=m_index_s（連續第 1 筆）
🔍 搜尋 前端工程師 第 2 頁
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8dyck?jobsource=m_index_s（連續第 1 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8ku4c?jobsource=m_index_s（連續第 2 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8725w?jobsource=m_index_s（連續第 1 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/85op5?jobsource=m_index_s（連續第 1 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/89saz?jobsource=m_index_s（連續第 2 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/76wll?jobsource=m_index_s（連續第 3 筆）
⏭️ 跳過重複職缺：https://www.104.com.tw/job/8e37k?jobsource=m_index_s（連續第 1 筆）
⏭️ 跳過重複職

In [12]:
import pandas as pd
import re

# 讀取 CSV
df = pd.read_csv('jobs.csv', encoding='utf-8', sep='|', engine='python')

def parse_salary(salary_str):
    if pd.isna(salary_str):
        return None
        
    # 去除所有空白和換行
    salary_str = str(salary_str).strip().replace('\n', '')
    
    # 找出第一個數字（包含逗號）
    matches = re.findall(r'[\d,]+', salary_str)
    if not matches:
        return None
        
    # 取第一個數字並移除逗號
    first_number = float(matches[0].replace(',', ''))
    
    # 判斷是否為年薪
    if '年薪' in salary_str:
        return int(first_number / 12)
    else:
        return int(first_number)

# 套用函數
df['final_salary'] = df['salary'].apply(parse_salary)

# 檢查結果
print(df[['salary', 'final_salary']].head())

# 儲存結果
df.to_csv('jobs.csv', encoding='utf-8-sig', index=False, sep='|')

              salary  final_salary
0   月薪40,000~60,000元         40000
1  月薪60,000~100,000元         60000
2        月薪40,000元以上         40000
3   月薪43,000~55,000元         43000
4   月薪40,000~70,000元         40000
